In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set a clean visual theme for all plots
sns.set_theme(style="whitegrid")

# Define the base directory where the bash/python script saved the outputs
base_dir = Path("wyniki/prompt_injection_eval")
all_data = []

# Dynamically load all CSVs and extract context from the directory structure
for csv_path in base_dir.rglob("benchmark_results.csv"):
    df_part = pd.read_csv(csv_path)
    
    # The folder structure is: base_dir / model_name / strategy_name / benchmark_results.csv
    strategy = csv_path.parent.name
    model = csv_path.parent.parent.name
    
    df_part['strategy'] = strategy
    df_part['model_folder'] = model
    all_data.append(df_part)

if all_data:
    df = pd.concat(all_data, ignore_index=True)
    print(f"Successfully loaded {len(df)} rows.")
    print(f"Models evaluated: {df['model_folder'].unique()}")
    print(f"Strategies tested: {df['strategy'].unique()}")
else:
    print("No CSV files found. Please check the base_dir path.")

In [ ]:
# Filter to only look at actual injection attempts (ignoring normal safe prompts)
df_attacks = df[df['ground_truth_attack'] == True]

def calculate_metrics(group):
    total = len(group)
    
    # Avoid zero-division errors if a specific strategy/model combo has no data
    if total == 0:
        return pd.Series({"ASR (%)": float('nan'), "Defense Rate (%)": float('nan'), "Total Attacks": 0})
    
    # ASR: Percentage of attacks that resulted in a 'harmful' classification
    # Multiplied by 100 to convert decimal to readable percentage
    asr = (group['effective_classification'] == 'harmful').sum() / total * 100
    
    # Defense: Percentage of attacks that were successfully rejected
    defense = (group['effective_classification'] == 'reject').sum() / total * 100
    
    return pd.Series({
        "ASR (%)": round(asr, 2),
        "Defense Rate (%)": round(defense, 2),
        "Total Attacks": total
    })

# Group by model and strategy, then apply the metrics function
metrics_table = df_attacks.groupby(['model_folder', 'strategy']).apply(calculate_metrics).reset_index()

display(metrics_table.sort_values(by=['model_folder', 'strategy']))

In [ ]:
# figsize=(14, 7) is chosen to provide enough horizontal width to fit 4 models 
# with 4 clustered bars each, without the labels overlapping.
plt.figure(figsize=(14, 7))

ax = sns.barplot(
    data=metrics_table, 
    x='model_folder', 
    y='ASR (%)', 
    hue='strategy',
    palette='magma'
)

plt.title('Attack Success Rate (ASR) by Model and Defense Strategy', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('ASR (% - lower is better)', fontsize=12)

# Move the legend entirely outside the plot to prevent it from covering the data bars
plt.legend(title='Strategy', bbox_to_anchor=(1.05, 1), loc='upper left')

# Add exact percentage labels on top of each bar for easy reading
for p in ax.patches:
    height = p.get_height()
    if pd.notnull(height) and height > 0:
        # xytext=(0, 3) pushes the text 3 pixels above the top of the bar
        ax.annotate(f'{height:.1f}%', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    fontsize=9, xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# Isolate the baseline strategy to see the model's natural vulnerabilities
df_baseline = df_attacks[df_attacks['strategy'] == '01_baseline']

if not df_baseline.empty:
    # Create a pivot table: Rows = Models, Columns = Attack Types, Values = ASR (%)
    pivot_asr = df_baseline.pivot_table(
        index='model_folder', 
        columns='attack_type', 
        values='effective_classification',
        aggfunc=lambda x: (x == 'harmful').sum() / len(x) * 100 if len(x) > 0 else 0
    )

    # figsize=(12, 6) gives a standard widescreen aspect ratio, ideal for heatmaps
    # where there are typically fewer models (rows) but many attack types (columns).
    plt.figure(figsize=(12, 6))
    
    # cmap="Reds" because higher ASR (red) is a worse security outcome
    sns.heatmap(pivot_asr, annot=True, fmt=".1f", cmap="Reds", cbar_kws={'label': 'ASR (%)'})
    
    plt.title('Vulnerability Heatmap by Attack Type (Baseline Strategy)', fontsize=14)
    plt.xlabel('Attack Type', fontsize=12)
    plt.ylabel('Model', fontsize=12)
    
    # Rotate X labels slightly so long attack type names don't crash into each other
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No baseline data found to generate the heatmap.")

In [ ]:
# Analyze the specific run where heuristic overrides were enabled
df_heuristics = df[df['strategy'] == '04_heuristic_defense'].copy()

if not df_heuristics.empty:
    # In pandas, empty tuples from your dataclass might be saved as strings like "()" or "None".
    # A length > 5 safely ensures it contains actual matched pattern text like "('ignore',)" 
    # instead of an empty tuple string, avoiding magic arbitrary checks.
    df_heuristics['any_heuristic_hit'] = (
        (df_heuristics['heuristic_injection_hits'].astype(str).str.len() > 5) |
        (df_heuristics['heuristic_command_hits'].astype(str).str.len() > 5) |
        (df_heuristics['heuristic_pii_hits'].astype(str).str.len() > 5)
    )
    
    # Create a confusion-matrix style table comparing Heuristic Flags to the Judge's Semantic Evaluation
    crosstab = pd.crosstab(
        index=df_heuristics['any_heuristic_hit'], 
        columns=df_heuristics['judge_classification'],
        rownames=['Heuristics Triggered?'],
        colnames=['Semantic Judge Classification'],
        margins=True,
        margins_name="Total"
    )
    
    print("Heuristic Detection vs LLM Judge Classification:")
    print("Use this to spot False Positives (Triggered=True but Judge=Accept) "
          "and False Negatives (Triggered=False but Judge=Harmful).")
    display(crosstab)
else:
    print("No heuristic strategy data found.")

In [ ]:
# Cell 6: Attack Success Rate (ASR) by Attack Type
# We use the baseline strategy to evaluate raw model vulnerability before heuristics/sanitization.
df_baseline = df_attacks[df_attacks['strategy'] == '01_baseline']

if not df_baseline.empty and 'attack_type' in df_baseline.columns:
    # Group by model and attack type, then calculate our standard metrics
    type_metrics = df_baseline.groupby(['model_folder', 'attack_type']).apply(calculate_metrics).reset_index()
    
    # Pivot the table to make it readable: Attack Types as rows, Models as columns
    pivot_type = type_metrics.pivot(index='attack_type', columns='model_folder', values='ASR (%)')
    
    # Drop any attack types that happened to have 0 examples in this limited run
    pivot_type = pivot_type.dropna(how='all')
    
    print("Attack Success Rate (ASR %) by Attack Type (Baseline Strategy):")
    # Apply a background gradient (Reds) so critical vulnerabilities instantly pop out visually
    display(pivot_type.style.background_gradient(cmap='Reds', axis=None).format("{:.1f}%", na_rep="-"))
else:
    print("No attack_type data available to analyze.")

In [ ]:
# Cell 7: Attack Success Rate (ASR) by Injected Task (Label)
if not df_baseline.empty and 'label' in df_baseline.columns:
    # Group by model and the specific task/label the attacker injected
    task_metrics = df_baseline.groupby(['model_folder', 'label']).apply(calculate_metrics).reset_index()
    
    # Pivot the table: Tasks as rows, Models as columns
    pivot_task = task_metrics.pivot(index='label', columns='model_folder', values='ASR (%)')
    
    # Clean up any empty rows from skipped tasks
    pivot_task = pivot_task.dropna(how='all')
    
    print("Attack Success Rate (ASR %) by Injected Task / Label (Baseline Strategy):")
    # Again, highlight higher numbers in red to flag dangerous compliance rates
    display(pivot_task.style.background_gradient(cmap='Reds', axis=None).format("{:.1f}%", na_rep="-"))
else:
    print("No task/label data available to analyze.")

In [ ]:
# Cell 8: Visualizing ASR by Attack Type Across Models
if not df_baseline.empty and 'attack_type' in df_baseline.columns:
    # figsize=(14, 6) provides a wide canvas, crucial when plotting many attack types on the X-axis
    plt.figure(figsize=(14, 6))
    
    # Use the pre-calculated type_metrics from Cell 6
    ax = sns.barplot(
        data=type_metrics,
        x='attack_type',
        y='ASR (%)',
        hue='model_folder',
        palette='viridis' # A distinct color palette to separate the models visually
    )
    
    plt.title('Vulnerability by Attack Type Across Models (Baseline Strategy)', fontsize=15)
    plt.xlabel('Attack Type', fontsize=12)
    plt.ylabel('Attack Success Rate (%)', fontsize=12)
    
    # Rotate X-axis labels 45 degrees so long attack types (e.g., 'instruction_override') don't overlap
    plt.xticks(rotation=45, ha='right')
    
    # Move the legend outside the chart to the upper left so it doesn't block the bars
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Add explicit percentage numbers to the top of each bar
    for p in ax.patches:
        height = p.get_height()
        if pd.notnull(height) and height > 0:
            # xytext=(0, 2) offsets the text exactly 2 points above the bar boundary
            ax.annotate(f'{height:.0f}%', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='bottom', 
                        fontsize=8, xytext=(0, 2), textcoords='offset points')
            
    plt.tight_layout()
    plt.show()